# LangChain RAG Agent

This notebook demonstrates how to build a traditional RAG (Retrieval-Augmented Generation) Agent using LangChain.

## 1. Import Required Libraries

In [1]:
import os
import re
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_milvus import Milvus
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
# PDF loader
from langchain_community.document_loaders import PyPDFLoader, TextLoader
# DashScope SDK - 必须导入并设置 API key
import dashscope

## 2. Load Environment Variables and Configuration

In [3]:
# ========== DashScope API Configuration ==========
# ⚠️ Important: Replace "YOUR_DASHSCOPE_API_KEY" below with your actual API key
# Example: key = "sk-1234567890abcdef..."
# PRIORITY: Code value > Environment variable
key = ""

# Only load .env if key is not set in code
if not key or key == "YOUR_DASHSCOPE_API_KEY":
    load_dotenv()  # Load .env file only if needed
    key = os.getenv("DASHSCOPE_API_KEY")
    if not key:
        raise ValueError(
            "❌ API key not set!\n"
            "Please replace 'YOUR_DASHSCOPE_API_KEY' in the code with your actual API key\n"
            "Or: Set DASHSCOPE_API_KEY=your_api_key in .env file"
        )
else:
    # Key is set in code, but still load .env for other variables (Milvus, etc.)
    load_dotenv()

# Clean API key (remove possible spaces and quotes)
if key:
    key = key.strip().strip('"').strip("'")

# base_url should be the API address, not the API key!
base_url = os.getenv("DASHSCOPE_API_BASE") or "https://dashscope.aliyuncs.com/compatible-mode/v1"

# Display API key information (for debugging)
print("=" * 60)
print("🔍 API Key Check")
print("=" * 60)
if key and key != "YOUR_DASHSCOPE_API_KEY":
    print(f"✅ API key is set")
    print(f"   Length: {len(key)} characters")
    print(f"   First 10 characters: {key[:10]}...")
    print(f"   Last 4 characters: ...{key[-4:]}")
    print(f"   Starts with 'sk-': {key.startswith('sk-')}")
    if not key.startswith('sk-'):
        print("   ⚠️ Warning: API key should usually start with 'sk-'")
    print(f"   base_url: {base_url}")
    # Check if .env file exists and might have conflicting key
    env_key = os.getenv("DASHSCOPE_API_KEY")
    if env_key and env_key != key:
        print(f"   ⚠️ Note: .env file has different key (ending with ...{env_key[-4:]})")
        print(f"   → Using code value (ending with ...{key[-4:]})")
else:
    print("❌ API key not set correctly!")
    print("   Please replace 'YOUR_DASHSCOPE_API_KEY' in the code with your actual API key")
print("=" * 60)

# ========== Milvus Connection Configuration ==========
# Method 1: Enter directly in code (recommended for testing)
milvus_uri = "https://xxxxxxxx.serverless.gcp-us-west1.cloud.zilliz.com"
milvus_user = ""
milvus_password = ""

# Method 2: Read from environment variables (if not set above, will try to read from .env file)
if not milvus_uri or milvus_uri == "YOUR_MILVUS_URI":
    milvus_uri = os.getenv("MILVUS_URI")
    milvus_user = os.getenv("MILVUS_USER")
    milvus_password = os.getenv("MILVUS_PASSWORD")


ValueError: ❌ API key not set!
Please replace 'YOUR_DASHSCOPE_API_KEY' in the code with your actual API key
Or: Set DASHSCOPE_API_KEY=your_api_key in .env file

## 3. Initialize LLM

In [ ]:
# Initialize LLM
graph_llm = ChatOpenAI(temperature=0, model_name="qwen-plus-2025-12-01", api_key=key, base_url=base_url)
llm = ChatOpenAI(temperature=0, model_name="qwen-plus", api_key=key, base_url=base_url)

## 4. Load Documents

Supports TXT and PDF formats

In [ ]:
# Step 1: Load documents
# Supports TXT and PDF formats
file_path = 'Cytokine Regulation and Function in T Cells.pdf'  # Can be changed to '../doc/company.pdf' to load PDF

# Select loader based on file extension
if file_path.endswith('.pdf'):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    print(f"PDF document loaded, {len(documents)} pages")
    total_chars = sum(len(doc.page_content) for doc in documents)
    print(f"Total content length: {total_chars} characters")
else:
    # Load TXT file
    loader = TextLoader(file_path, encoding='utf-8')
    documents = loader.load()
    print(f"TXT document loaded, content length: {len(documents[0].page_content)} characters")

## 5. Text Splitting

In [ ]:
# Step 2: Text splitting and cleaning
def clean_text(text):
    """Clean text, remove characters that may cause encoding issues"""
    if not text:
        return ""
    # Remove control characters (except newline, tab, and carriage return)
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    # Remove zero-width characters
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e\u2060-\u206f]', '', text)
    # Ensure text can be properly encoded as UTF-8
    try:
        text.encode('utf-8')
    except UnicodeEncodeError:
        # If there are still issues, use error handling
        text = text.encode('utf-8', errors='ignore').decode('utf-8')
    return text

# Clean original documents first
for doc in documents:
    doc.page_content = clean_text(doc.page_content)
    # Clean metadata
    if doc.metadata:
        cleaned_metadata = {}
        for key, value in doc.metadata.items():
            if isinstance(value, str):
                cleaned_metadata[key] = clean_text(value)
            else:
                cleaned_metadata[key] = value
        doc.metadata = cleaned_metadata

chunk_size = 250
chunk_overlap = 30
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(documents)
print(f"Document split into {len(splits)} chunks")

# Clean split documents again and filter empty documents
cleaned_splits = []
for doc in splits:
    cleaned_content = clean_text(doc.page_content)
    if cleaned_content.strip():  # Only keep non-empty documents
        doc.page_content = cleaned_content
        cleaned_splits.append(doc)

splits = cleaned_splits
print(f"Remaining {len(splits)} valid chunks after cleaning")

## 6. Initialize Embeddings

In [ ]:
# Step 3: Initialize Embeddings
# ⚠️ 重要：根据 DashScope 官方文档，必须先设置 dashscope.api_key
# 参考文档：https://help.aliyun.com/zh/model-studio/text-embedding-synchronous-api

# 使用前面 Cell 4 中已经设置的 key（如果前面的 cell 已运行，key 已存在）
# 如果前面的 cell 没有运行，这里作为备用方案重新获取
if 'key' not in globals() or not key or len(key) < 20:
    load_dotenv()
    key = os.getenv("DASHSCOPE_API_KEY")
    if key:
        key = key.strip().strip('"').strip("'")
    else:
        raise ValueError("❌ API key 未设置！请先运行 Cell 4 设置 API key")

# 设置 dashscope.api_key（这是关键步骤！）
dashscope.api_key = key

print("=" * 60)
print("🔧 DashScope API Key Configuration")
print("=" * 60)
print(f"✅ dashscope.api_key 已设置")
print(f"   Key 长度: {len(key)} 字符")
print(f"   Key 前缀: {key[:10]}...")
print(f"   Key 后缀: ...{key[-4:]}")

# 使用 DashScope SDK 直接测试 API（按照官方文档方式）
print("=" * 60)
print("🧪 Testing API Key with DashScope SDK (官方文档方式)")
print("=" * 60)
from http import HTTPStatus

try:
    # 按照 DashScope 官方文档的方式直接调用
    resp = dashscope.TextEmbedding.call(
        model="text-embedding-v4",
        input="测试文本",
        dimension=1024
    )
    
    if resp.status_code == HTTPStatus.OK:
        print("✅ API key 测试成功！DashScope SDK 调用正常")
        print(f"   返回向量维度: {len(resp.output['embeddings'][0]['embedding'])}")
    else:
        print(f"❌ API 调用失败: status_code={resp.status_code}")
        print(f"   code: {resp.code}")
        print(f"   message: {resp.message}")
        raise ValueError(f"DashScope API 调用失败: {resp.message}")
        
except Exception as e:
    print(f"❌ API key 测试失败: {e}")
    print("\n💡 可能的原因:")
    print("1. API key 无效或已过期")
    print("2. API key 格式不正确")
    print("3. 账户未完成验证")
    print("4. 账户余额不足或服务未激活")
    print("\n🔧 解决方案:")
    print("1. 访问 https://dashscope.console.aliyun.com")
    print("2. 完成账户验证")
    print("3. 检查 API key 是否有效")
    print("4. 确保账户有足够余额")
    print("=" * 60)
    raise ValueError("API key 无效，请解决上述问题后再继续")

print("=" * 60)

# 现在使用 LangChain 的 DashScopeEmbeddings
print("=" * 60)
print("🔧 初始化 LangChain DashScopeEmbeddings")
print("=" * 60)
try:
    test_embeddings = DashScopeEmbeddings(
        model="text-embedding-v4",
        dashscope_api_key=key
    )
    # 再次测试
    test_result = test_embeddings.embed_query("test")
    print("✅ LangChain DashScopeEmbeddings 初始化成功")
    print(f"   测试结果维度: {len(test_result)}")
except Exception as e:
    print(f"⚠️ LangChain DashScopeEmbeddings 测试失败: {e}")
    print("   但 DashScope SDK 已测试成功，可以继续使用")
print("=" * 60)


# Create a safe Embeddings wrapper
# Try different import methods
try:
    from langchain_core.embeddings import Embeddings
except ImportError:
    try:
        from langchain.embeddings.base import Embeddings
    except ImportError:
        # If all imports fail, use ABC base class
        from abc import ABC, abstractmethod
        from typing import List as TypingList
        
        class Embeddings(ABC):
            @abstractmethod
            def embed_documents(self, texts: TypingList[str]) -> TypingList[TypingList[float]]:
                """Embed document list"""
                pass
            
            @abstractmethod
            def embed_query(self, text: str) -> TypingList[float]:
                """Embed query text"""
                pass

from typing import List

class SafeDashScopeEmbeddings(Embeddings):
    """Wrapper for DashScopeEmbeddings to ensure text is safe before sending"""
    def __init__(self, base_embeddings):
        self.base_embeddings = base_embeddings
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Embed document list"""
        # Clean text again before sending
        cleaned_texts = [clean_text(text) for text in texts]
        # Filter empty texts
        cleaned_texts = [t for t in cleaned_texts if t.strip()]
        if not cleaned_texts:
            return []
        # Process in batches to avoid processing too much at once
        all_embeddings = []
        batch_size = 5  # Smaller batch size
        for i in range(0, len(cleaned_texts), batch_size):
            batch = cleaned_texts[i:i+batch_size]
            try:
                batch_embeddings = self.base_embeddings.embed_documents(batch)
                all_embeddings.extend(batch_embeddings)
            except Exception as e:
                print(f"Error processing batch {i//batch_size + 1}: {e}")
                # If batch fails, try processing one by one
                for text in batch:
                    try:
                        emb = self.base_embeddings.embed_documents([text])
                        all_embeddings.extend(emb)
                    except Exception as e2:
                        print(f"Skipping unprocessable text: {text[:50]}... Error: {e2}")
                        # Add zero vector as placeholder
                        all_embeddings.append([0.0] * 1536)  # DashScope default dimension
        return all_embeddings
    
    def embed_query(self, text: str) -> List[float]:
        """Embed query text"""
        cleaned_text = clean_text(text)
        return self.base_embeddings.embed_query(cleaned_text)

# Initialize base embeddings (using verified key)
# 确保 dashscope.api_key 已设置（已在上面设置）
base_embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=key
)

# Use safe wrapper
embeddings = SafeDashScopeEmbeddings(base_embeddings)
print("✅ Embeddings model initialized (with safe wrapper)")

## 7. Build Vector Index and Store to Milvus

In [ ]:
# Step 4: Build vector index and store to Milvus
vectorstore = Milvus.from_documents(
    documents=splits,
    collection_name="company_milvus",
    embedding=embeddings,
    connection_args={
        "uri": "https://in03-13d3fc765a723cc.serverless.gcp-us-west1.cloud.zilliz.com",
        "user": "db_13d3fc765a723cc",  # Replace with your own
        "password": "Ew7|K2USgunQOqnb",
    }
)
print("Vector index built and stored to Milvus")

## 8. Create RAG Chain

In [ ]:
# Step 5: Create RAG Chain (Immunology Experiment Assistant - interactive)
prompt = PromptTemplate(
    template="""You are an immunology experiment-planning assistant.
Design an executable experimental plan using ONLY the provided context. Do NOT invent parameters (e.g., concentrations, incubation times, catalog numbers, instrument models) unless explicitly stated in the context.

Rules:
1) If the context has relevant info, propose a minimal, actionable plan tailored to the goal.
2) If critical details are missing, ask up to 3 clarifying questions (only the most critical).
3) Keep the response concise, but prioritize actionability over being short.

Question: {question}
Context: {context}

Answer in this format:
- Goal:
- Hypothesis:
- Minimal plan (3-7 steps):
- Controls:
- Readouts:
- Missing critical info (if any):
- Clarifying questions (0-3):""",
    input_variables=["question", "context"],
)

rag_chain = prompt | graph_llm | StrOutputParser()
print("RAG Chain 已创建")

## 9. Test RAG Chain

In [ ]:
# Step 6: Test RAG Chain (Experiment-Assistant-friendly + Debuggable)
question = "What CD4+ T helper subsets are discussed in this article?"

# 1) Retrieve more chunks for better coverage
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})  # 5~10都可以
docs = retriever.invoke(question)

# 2) Basic debug: show what was retrieved (with safer preview)
print("=" * 70)
print(f"Retrieved {len(docs)} document chunks")
print("=" * 70)

if not docs:
    print("No documents retrieved.")
else:
    for i, doc in enumerate(docs[:5], 1):  # show first 5 chunks
        preview = doc.page_content.replace("\n", " ")
        preview = (preview[:300] + "...") if len(preview) > 300 else preview
        meta = getattr(doc, "metadata", {}) or {}
        print(f"\nChunk {i} | len={len(doc.page_content)} | meta={meta}")
        print(f"Preview: {preview}")

print("=" * 70)

# 3) Deduplicate + build context (avoid repeated chunks)
seen = set()
unique_texts = []
for doc in docs:
    text = (doc.page_content or "").strip()
    if not text:
        continue
    key = text[:200]  # simple signature for near-duplicate filtering
    if key in seen:
        continue
    seen.add(key)
    unique_texts.append(text)

# 4) Truncate total context to avoid overlong prompts (MVP-safe)
MAX_CONTEXT_CHARS = 6000  # adjust if needed
context = "\n\n".join(unique_texts)
context = context[:MAX_CONTEXT_CHARS]

# 5) Run RAG Chain
generation = rag_chain.invoke({"context": context, "question": question})

print("\nGenerated answer:")
print(generation)

## 10. Define AgentState

In [ ]:
# Step 7: Define AgentState
class AgentState(MessagesState):
    next: str

## 11. Create Traditional RAG Agent Node

In [ ]:
# Step 8: Create traditional RAG Agent node (Experiment Assistant)
def vec_kg(state: AgentState):
    last_msg = state["messages"][-1]
    question = last_msg.content

    # --- Experiment-assistant prompt (interactive, allow up to 3 clarifying questions) ---
    prompt = PromptTemplate(
        template="""You are an immunology experiment-planning assistant.
Design an executable experimental plan using ONLY the retrieved context. Do NOT invent parameters (e.g., concentrations, incubation times, catalog numbers, instrument models) unless explicitly stated in the context.

Rules:
1) If the context has relevant info, propose a minimal, actionable plan tailored to the goal.
2) If critical details are missing, ask up to 3 clarifying questions (only the most critical).
3) Keep the response concise, but prioritize actionability over being short.

Question: {question}
Context: {context}

Answer in this format:
- Goal:
- Hypothesis:
- Minimal plan (3-7 steps):
- Controls:
- Readouts:
- Missing critical info (if any):
- Clarifying questions (0-3):""",
        input_variables=["question", "context"],
    )

    rag_chain = prompt | graph_llm | StrOutputParser()

    # --- Retriever: get more coverage than k=1 ---
    retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
    docs = retriever.invoke(question)

    # --- docs -> clean context text (dedup + truncate) ---
    seen = set()
    unique_texts = []
    for d in docs:
        text = (d.page_content or "").strip()
        if not text:
            continue
        sig = text[:200]
        if sig in seen:
            continue
        seen.add(sig)
        unique_texts.append(text)

    context = "\n\n".join(unique_texts)

    MAX_CONTEXT_CHARS = 6000
    context = context[:MAX_CONTEXT_CHARS]

    generation = rag_chain.invoke({"context": context, "question": question})

    final_response = [HumanMessage(content=generation, name="vec_kg")]
    return {"messages": final_response}

## 12. Test vec_kg Node

In [ ]:
# Step 9: Test vec_kg node (Experiment Assistant)
test_state = AgentState(
    messages=[
        HumanMessage(
            content="Based on this article, design a minimal experiment to study CD4+ T helper cell differentiation."
        )
    ]
)

result = vec_kg(test_state)

print("RAG Agent response:")
for msg in result["messages"]:
    print("-" * 60)
    print(msg.content)